# Chapter 9 Exercise: Coffee Maker Coverage and Sufficiency

Produce a real coverage report over your own coffee-maker model, apply
Hawkins' sufficiency reading to two of your own already-built judgment
records, and check both for staleness against one real edit, mirroring
`chapters/ch09-coverage-sufficiency/{01-requirement-coverage,
02-evidence-completeness,03-stale-detection}.ipynb` one-for-one, for the
coffee-maker domain instead of the toaster's own.

## Problem

This chapter adds no new model element, in either domain: main Chapter 9
queries `models/ch08-cumulative.sysml` directly rather than growing a
`models/ch09-cumulative.sysml` that would carry nothing new. This exercise
does the same, over your own accumulated coffee-maker model from Chapters
1-8, with adaptations the exercise track needs that the main chapters do
not, since exercises never commit a solution file (no answer key):

- **No committed model fixture.** Main Chapter 9 loads a real, committed
  `models/ch08-cumulative.sysml`. Step 1 below is a blank placeholder:
  paste your own full, completed Chapters 1-8 coffee-maker model as
  `source`, the same way `exercises/ch08/exercise.ipynb`'s own Step 1 asks
  you to paste your completed Chapter 7 model.
- **Two placeholders, not one committed pair of files.** Main Chapter 9
  hashes `AS-C06` against `models/ch06-cumulative.sysml` (the model it was
  actually written against) and `AS-C08` against `models/ch08-cumulative.sysml`
  (the current model): two different, separately committed files. This
  track has no such committed pair, so Step 2 below asks for a SECOND
  placeholder, `ch06_source`: paste your own `exercises/ch06/exercise.ipynb`
  Step 2 `source_with_decomp` (the model state `AS-C06-EX` was actually
  written against, before `Impeller`, before Chapter 7, before Chapter 8) --
  a genuinely separate variable from `source` above, mirroring how real
  `03-stale-detection.ipynb` loads `ch06_source = Path("../../models/
  ch06-cumulative.sysml").read_text()` as its own variable, not a slice of
  the current model. `AS-C06-EX` is hashed against `ch06_source`;
  `AS-C08-EX` is hashed against `source`: the same split real Chapter 9
  uses, each record hashed against the file it was actually written
  against.
- **Records are reconstructed as scaffolds, not finished text.** `AS-C06-EX`
  and `AS-C08-EX` are reconstructed below exactly as `exercises/ch06/
  exercise.ipynb` and `exercises/ch08/exercise.ipynb` themselves declare
  them. `AS-C06-EX`'s own fixed fields (real, not hints): `identifier`,
  `kind`, `model_ref` (`CoffeeDemo::WaterMover`), `scope`
  (`CoffeeDemo::WaterMover and its realizations`), `premises` (two real,
  filled entries -- a confirmed model fact plus a domain premise),
  `evidence_refs` (resolved live from `model.find()`), `disposition`,
  `dependency_freshness`, `engineering_conclusion` (`"undetermined"`), and
  `record_kind`; its hints (fill in your own argued text): `claim`,
  `criteria`, `assumption_refs`, `rationale`, `counterevidence`, and
  `residual_uncertainties`. `AS-C08-EX`'s own fixed fields: `identifier`,
  `kind`, `model_ref` (`CoffeeDemo::deliveredMassBoundedBySupply`),
  `premises` and `assumption_refs` (both genuinely empty `[]`, the same way
  real `AS-C08`'s are -- a deductive, Z3-proved claim needs no separate
  premise beyond the proof itself), `disposition`, `dependency_freshness`,
  `engineering_conclusion` (`"supported"`), and `record_kind`; its hints:
  `claim`, `scope`, `criteria`, `rationale`, `counterevidence`, and
  `residual_uncertainties`. `AS-C08-EX`'s `evidence_refs` is a special case:
  in `exercises/ch08/exercise.ipynb`'s own notebook it is computed live from
  a real `verify_holds()` Z3 call, not a hint there at all -- it is left as
  a hint HERE only because re-running that solver machinery would be
  disproportionate to what this step demonstrates; paste your own printed
  result from your Chapter 8 exercise in its place.

Use only constructs and operations introduced up to Chapter 9.

## Step 1: A real coverage report over your own model

Chapters 1 through 8 declared satisfy claims one at a time. This step asks
a different question across your whole model at once: for every
requirement usage your model declares, has anyone actually claimed a
candidate satisfies it, and of which polarity?

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# Paste your own completed Chapters 1-8 coffee-maker model here (the same
# full source your own Chapter 8 exercise already builds on). There is no
# committed exercises/models/*.sysml fixture to load instead -- exercises
# never commit a solution file, so this is the same blank placeholder every
# earlier chapter's own Step 1 cell already asks you to fill.
source = """
# Your solution here
"""

model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"


The model loads cleanly. Before querying it, the same language-tier
control every chapter carries: a `satisfy` naming a requirement that was
never declared still fails to load.

In [ ]:
# A satisfy claim naming a requirement that doesn't exist fails to load; it can
# never silently show up as a "claim" this exercise's coverage query would count.
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def Widget { attribute x : Real default = 1.0; }
    part w : Widget;
    assert satisfy missingReq by w;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for an undeclared requirement"
print(f"Negative control ok: bad.ok={bad.ok}")


With the model loaded, the coverage report starts from the same two
surfaces every chapter's own satisfy claim already used: every named
`RequirementUsage` from `model.query()`, and every `SatisfyRequirementUsage`,
named or not, from `get_satisfy_relationships()` (D-001: `model.query()`
returns none of these directly). Each raw satisfy element carries `subsets`
(the requirement it claims against), `subject` (the candidate, when there
is one), `isNegated` (a positive or a negative claim) and
`sysx:declaredKeyword` (`"verify"` for a verification-case objective).

In [ ]:
from toaster.query import find_requirements, get_satisfy_relationships, ApiIndex

idx = ApiIndex(model)
requirements = sorted(r.id for r in find_requirements(model))
raw_satisfies = get_satisfy_relationships(model)

claims = []
for s in raw_satisfies:
    claims.append({
        "id": s.get("qualifiedName"),
        "requirement": idx.qn(s["subsets"]) if "subsets" in s else None,
        "subject": idx.qn(s["subject"]) if "subject" in s else None,
        "is_verify": s.get("sysx:declaredKeyword") == "verify",
        "is_negated": bool(s.get("isNegated", False)),
    })

print(f"requirements: {requirements}")
for c in claims:
    print(c)


Joining the real claims by requirement, split by polarity, is the actual
coverage report; a verification-case objective (`is_verify=True`) names a
requirement without claiming anything about a subject, so it is reported
separately, not folded into either side.

In [ ]:
from collections import defaultdict

coverage = {r: {"positive": [], "negative": [], "verify_objectives": []} for r in requirements}
dropped = []
for c in claims:
    if c["requirement"] not in coverage:
        dropped.append(c)
        continue
    if c["is_verify"]:
        coverage[c["requirement"]]["verify_objectives"].append(c["id"])
    elif c["is_negated"]:
        coverage[c["requirement"]]["negative"].append(c["subject"])
    else:
        coverage[c["requirement"]]["positive"].append(c["subject"])

print(f"claims dropped (requirement not in the named list above): {len(dropped)}")
assert dropped == [], f"Unexpected dropped claims: {dropped}"

for r in requirements:
    cov = coverage[r]
    positive, negative, verify_objectives = cov["positive"], cov["negative"], cov["verify_objectives"]
    covered = bool(positive)
    print(f"{r}:")
    print(f"  positive satisfy claims: {positive}")
    print(f"  negative satisfy claims: {negative}")
    print(f"  verify objectives (no subject): {verify_objectives}")
    print(f"  covered (has >=1 positive claim): {covered}")

# Confirmed empirically against a real, assembled Chapters 1-8 coffee model
# (not assumed): tempCheck has never had a positive claim -- only hot's own
# failing claim, plus BrewTempTest's own verify objective (no subject) --
# mirroring timely's exact one-sided coverage shape.
assert coverage["CoffeeDemo::tempCheck"]["positive"] == []
assert coverage["CoffeeDemo::tempCheck"]["negative"] == ["CoffeeDemo::hot"]

# brewReq's own positive coverage: exercises/ch06/exercise.ipynb's own Step 4
# only explicitly writes `assert not satisfy brewReq by weak;` -- a positive
# claim on rated is implied by mirroring rated/weak, not spelled out as its
# own explicit assertion. Report what your own model actually shows, rather
# than assuming a specific candidate name -- the same "don't assume, verify
# empirically" discipline exercises/ch08/exercise.ipynb's own Step 2 states
# explicitly ("report what you actually find ... it may or may not
# [match]").
print(f"brewReq positive satisfy claims found: {coverage['CoffeeDemo::brewReq']['positive']}")
assert coverage["CoffeeDemo::brewReq"]["positive"], (
    "Expected at least one positive satisfy claim against brewReq"
)
assert coverage["CoffeeDemo::brewReq"]["negative"] == ["CoffeeDemo::weak"]

`brewReq` is checked on both sides: `rated` was checked and found to
satisfy it, `weak` was checked and found not to. `tempCheck` has only ever
been checked against a candidate that fails it (`hot`), plus
`BrewTempTest`'s own verify objective, which claims nothing about a
subject: `nominal`, the usage meant to represent a brew unit actually
meeting its temperature requirement, has no `assert satisfy tempCheck by
nominal` anywhere in the model. This does **not** mean `nominal` fails
`tempCheck`: its `brewTemp` carries no value at all, so no one has ever
actually checked whether `nominal` satisfies `tempCheck` in the first
place. The gap this query finds is an absence of a claim, not evidence of
a failed one. This is exactly the reading real Chapter 9 gives `timely`.

Is this join trustworthy, or could a differently-written join reach a
different answer? First, a genuine wrong way to join the same data: a join
that counts ANY claim, positive or negative, as coverage.

In [ ]:
# A polarity-blind join: exactly the mistake of counting a FAILING claim as coverage.
naive_covered = defaultdict(list)
for c in claims:
    if c["requirement"] and c["subject"]:
        naive_covered[c["requirement"]].append(c["subject"])

print("A polarity-blind join would report:")
for r in requirements:
    claimants = naive_covered.get(r, [])
    print(f"  {r}: covered={bool(claimants)}  claimed by={claimants}")

assert naive_covered["CoffeeDemo::tempCheck"] == ["CoffeeDemo::hot"]


A polarity-blind join reports `tempCheck` as covered, using `hot`'s own
FAILING claim as the evidence, exactly backwards. `src/toaster/query.py`
already ships a `requirement_coverage()` helper for exactly this kind of
question (the same helper in which main Chapter 9 found and fixed this
exact bug); confirm it agrees with the join built above rather than
inventing a second, untested mechanism.

In [ ]:
from toaster.query import requirement_coverage

repo_coverage = {c["requirement"]: c for c in requirement_coverage(model)}
for r in requirements:
    print(repo_coverage[r])

assert repo_coverage["CoffeeDemo::tempCheck"]["covered"] is False
assert repo_coverage["CoffeeDemo::tempCheck"]["satisfied_by"] == []
assert repo_coverage["CoffeeDemo::tempCheck"]["failed_by"] == ["CoffeeDemo::hot"]
assert repo_coverage["CoffeeDemo::brewReq"]["covered"] is True
# Same "don't assume, verify empirically" discipline as the hand-built join
# above: report what satisfied_by actually names, rather than asserting a
# specific candidate identifier.
print(f"brewReq satisfied_by (repo helper): {repo_coverage['CoffeeDemo::brewReq']['satisfied_by']}")
assert repo_coverage["CoffeeDemo::brewReq"]["satisfied_by"]
assert repo_coverage["CoffeeDemo::brewReq"]["failed_by"] == ["CoffeeDemo::weak"]

# The two independently-written joins -- hand-built above and this
# repository helper -- must agree on WHICH candidates cover brewReq, not
# just on covered/not-covered.
assert repo_coverage["CoffeeDemo::brewReq"]["satisfied_by"] == coverage["CoffeeDemo::brewReq"]["positive"]

Two independently-written joins over the same raw data, one built cell
by cell above and one already living in the repository's own query module,
agree exactly, and only because both actually track polarity: the
deliberately wrong join above reaches the opposite answer for `tempCheck`.
That is what "coverage" has to mean for this check to be worth anything at
all: a positive claim someone actually made, not merely a claim of any
kind.

## Step 2: Evidence sufficiency, applied to your own two records

This tutorial has no central registry of every `ReviewRecord` it has ever
built: each chapter constructs its own records as local Python objects,
with nothing shared beyond the file each one lives in. This step
reconstructs two records you already built: `AS-C06-EX`
(`exercises/ch06/exercise.ipynb`, a mechanism-selection judgment) and
`AS-C08-EX` (`exercises/ch08/exercise.ipynb`, grounded in your own Chapter 8
Z3 proof), and applies Hawkins' sufficiency check
(`uv run python -m glossary tutorial sufficiency`) to each. What follows
demonstrates the mechanics of the check on these two records specifically,
not an exhaustive audit of every judgment record this exercise track has
ever produced.

Before reconstructing the two records, the mechanized half of a
sufficiency check: `validate_record()` already refuses a record whose
`counterevidence` field is empty outright, distinct from asking whether a
populated field is substantive, which this step goes on to ask.

In [ ]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

blank_counterevidence = ReviewRecord(
    identifier="AS-BAD-EX",
    kind="asserted_solution",
    claim="Some claim",
    model_ref="CoffeeDemo",
    content_hash=hash_content(source),
    scope="Some scope",
    criteria="Some criteria",
    rationale="Some rationale",
    counterevidence="",   # intentionally empty
    record_kind="worked_example",
)
errors = validate_record(blank_counterevidence)
assert len(errors) > 0, "Expected validation to fail on empty counterevidence"
print(f"Negative control ok: errors={errors}")


`AS-C06-EX`, reconstructed below exactly as `exercises/ch06/
exercise.ipynb` declares it: its own `premises` are real, filled text (not
a hint): paste your own filled prose into `claim`, `criteria`,
`assumption_refs`, `rationale`, `counterevidence` and `residual_uncertainties`
in place of the `"# ..."` hints below. Its `content_hash` is computed
against `ch06_source` (built in the next cell): your own
`exercises/ch06/exercise.ipynb` Step 2 `source_with_decomp`, the model
state `AS-C06-EX` was actually written against -- a genuinely separate
variable from `source` (this exercise's own Step 1), the same way real
`AS-C06` is hashed against `models/ch06-cumulative.sysml`, not against the
current model.

In [ ]:
# Paste your own exercises/ch06/exercise.ipynb Step 2 source_with_decomp
# here -- the model state AS-C06-EX was actually written against, before
# Impeller (ch06 Step 4), before Chapter 7, before Chapter 8. A genuinely
# separate variable from `source` above (this exercise's own Step 1),
# mirroring how real 03-stale-detection.ipynb loads
# ch06_source = Path("../../models/ch06-cumulative.sysml").read_text() as
# its own variable, not a slice of the current model.
ch06_source = """
# Your solution here (paste your own exercises/ch06 Step 2 source_with_decomp -- the
# model state AS-C06-EX was actually written against, before Impeller/ch07/ch08)
"""

In [ ]:
# Confirm ch06_source is genuinely your own ch06-stage model, not left blank
# and not accidentally a copy of the current (ch08-stage) model: a check that
# only asserted "the string changed" would pass even on an unfilled or wrong
# placeholder, the exact failure mode this exercise track already caught and
# fixed once for the growth-diff cell below (DEFERRED.md D-030's own
# "reliesOnSibling" lesson, ch08 exercise).
ch06_model = conn.load_from_content(ch06_source, strict=False)
assert ch06_model.ok, f"ch06_source failed to load: {format_diagnostics(ch06_model.diagnostics)}"
assert ch06_model.find("CoffeeDemo::WaterMover") is not None, (
    "ch06_source must declare WaterMover -- paste your own ch06 Step 2 model, not a blank string"
)
assert "transferEfficiency" not in ch06_source, (
    "ch06_source still contains transferEfficiency (a Chapter 7 addition) -- "
    "paste your ch06 Step 2 source_with_decomp, before Impeller/ch07/ch08, "
    "not your current (ch08-stage) model"
)
print("ch06_source confirmed: loads cleanly, declares WaterMover, predates transferEfficiency.")

In [ ]:
brew_start = model.find("CoffeeDemo::BrewStart")
brew_cancel = model.find("CoffeeDemo::BrewCancel")
print(f"BrewStart: {brew_start.kind}, BrewCancel: {brew_cancel.kind}")

as_c06_ex = ReviewRecord(
    identifier="AS-C06-EX",
    kind="asserted_solution",
    claim="# Which mechanism WaterMover commits to, and which plausible alternative it is selected over",
    model_ref="CoffeeDemo::WaterMover",
    content_hash=hash_content(ch06_source),
    scope="CoffeeDemo::WaterMover and its realizations",
    criteria="# What the chosen mechanism must be able to do: respond to a discrete control signal, expose a rating brewReq can check",
    premises=[
        "CoffeeDemo declares BrewStart and BrewCancel (Chapter 4), "
        "discrete signals marking the start and cancellation of a "
        "brewing cycle, confirmed by model.find() above.",
        "Domain premise, not derived from the model: a motor-driven "
        "impeller responds to being switched on and off directly, "
        "while a gravity-fed drip mechanism has no actuation point of "
        "its own and needs a separate valve or timing mechanism to "
        "start or stop the flow on command. Neither WaterMover nor any "
        "of its realizations is yet connected to a control signal in "
        "this model; this premise is about the physical mechanisms "
        "themselves, not about what the model currently wires "
        "together.",
    ],
    assumption_refs=["# What WaterMover itself does NOT already commit to -- this record is what actually chooses the mechanism"],
    evidence_refs=[
        f"model.find('CoffeeDemo::BrewStart') resolves to a real {brew_start.kind}, confirmed above.",
        f"model.find('CoffeeDemo::BrewCancel') resolves to a real {brew_cancel.kind}, confirmed above.",
    ],
    rationale="# Why this mechanism is preferred, argued from how the two mechanisms work -- not from anything the model currently wires together",
    counterevidence="# What this selection does not rule out, and what stays unmodeled (mirror AS-C06's own challenge about Joule heating)",
    residual_uncertainties="# What would let this selection be revisited against a real trade study",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(as_c06_ex)
print(f"AS-C06-EX validation errors: {errors}")


`AS-C08-EX`, reconstructed the same way from `exercises/ch08/
exercise.ipynb` Step 2. Its own `premises` and `assumption_refs` are
genuinely empty there too, the same way real `AS-C08`'s are (a deductive,
Z3-proved claim needs no separate premise beyond the proof itself). Its
`evidence_refs` entry, unlike `AS-C06-EX`'s, is left as a hint rather than
recomputed here: the real value is the `verify_holds()` result string your
own Chapter 8 exercise already printed against a Z3 companion file, and
reproducing that machinery here (a solver, a scratch companion file) would
be disproportionate to what this step demonstrates. Paste your own
printed result in its place.

In [ ]:
as_c08_ex = ReviewRecord(
    identifier="AS-C08-EX",
    kind="asserted_solution",
    claim=(
        "# deliveredMassBoundedBySupply, a hand-restated real-arithmetic "
        "lemma of the same shape as WaterMover's transferEfficiencyBounded "
        "constraint and deliveredMass's own definition, holds for every "
        "value of transferEfficiency in [0,1] and every non-negative "
        "throughput and duration a companion restatement admits"
    ),
    model_ref="CoffeeDemo::deliveredMassBoundedBySupply",
    content_hash=hash_content(source),
    scope=(
        "# The lemma governs the companion restatement's own "
        "moverCheck.transferEfficiency, moverCheck.throughput and "
        "moverCheckDuration features; state plainly that it is not a "
        "solver-checked reference to WaterMover's own "
        "transferEfficiencyBounded or deliveredMass (DEFERRED.md D-030, D-031)"
    ),
    criteria=(
        "# What verify_holds() must report for this to count as proved: a "
        "single satisfied verdict, with z3 named in the reason text, not "
        "propagation alone"
    ),
    premises=[],
    assumption_refs=[],
    evidence_refs=[
        "# paste your own verify_holds() result string from "
        "exercises/ch08/exercise.ipynb Step 2 here (the pos_verdict printed there)"
    ],
    rationale=(
        "# Why this is a materially different kind of evidence from "
        "verify_satisfaction()'s point evaluation, and why it is a "
        "narrower claim than proving WaterMover's own conservation property"
    ),
    counterevidence=(
        "# State plainly: this proof is NOT a solver-checked reference to "
        "WaterMover's own transferEfficiencyBounded or deliveredMass "
        "(DEFERRED.md D-030, D-031) -- confirmed directly in your own "
        "Chapter 8 exercise: a sibling constraint (reliesOnSibling) that "
        "depends entirely on transferEfficiencyBounded's own stated bound, "
        "without restating it, is left undecided with a genuine witness "
        "violating that very bound, because this toolchain's Z3 backend "
        "never composes two separately declared assert constraints. Also "
        "cite D-029: the companion file is used instead of the committed "
        "model directly because toaster.modelcheck's own text parser "
        "cannot yet read a verdict line for a constraint that is also the "
        "subject of an assert satisfy declaration."
    ),
    residual_uncertainties=(
        "# What this proof does not establish: whether transferEfficiency, "
        "throughput and duration ever take values outside the bound in a "
        "real candidate; whether the restated copy still matches "
        "WaterMover's own construct if either is edited later; whether any "
        "physical impeller instance's transferEfficiency/throughput values, "
        "beyond rated/weak, have been checked against this property -- "
        "WaterMover itself remains an abstract carrier with no concrete "
        "realization of its own"
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

errors = validate_record(as_c08_ex)
print(f"AS-C08-EX validation errors: {errors}")


Both records validate cleanly. Hawkins' sufficiency idea is about whether
stated PREMISES make a conclusion's probable truth follow, so the premises
themselves belong in this check, not only `counterevidence` and
`residual_uncertainties`. `AS-C06-EX`'s own `premises` entries are exactly
what its own selection argument rests on (a confirmed model fact, plus a
domain premise about how the two mechanisms work); `AS-C08-EX`'s `premises`
and `assumption_refs` are both genuinely empty.

In [ ]:
print(f"AS-C06-EX premises: {as_c06_ex.premises}")
print(f"AS-C06-EX assumption_refs: {as_c06_ex.assumption_refs}")
print()
print(f"AS-C08-EX premises: {as_c08_ex.premises}")
print(f"AS-C08-EX assumption_refs: {as_c08_ex.assumption_refs}")

Is `AS-C08-EX`'s empty `premises` field itself a sufficiency concern? Read
against what the record actually claims, no: `AS-C06-EX`'s claim is an
inductive engineering judgment (a mechanism selected FOR stated reasons,
reasons that could be wrong), exactly what Hawkins' idea is about, so it
needs premises to carry the inductive weight. `AS-C08-EX`'s claim, narrowly
read (see your own `claim` and `scope` above), is that a stated lemma was
PROVED by Z3 -- a deductive result, not an inductive one, so there is no
separate premise beyond the proof itself for the record to name;
`evidence_refs` carries the proof, and that is the right place for it to
live. The genuine risk is not the empty `premises` field but a claim that
quietly outgrows its own narrow scope -- which is exactly what
`AS-C08-EX`'s own `counterevidence` exists to prevent, by refusing to let
the proved lemma stand in for a claim about `WaterMover`'s real
`transferEfficiencyBounded` and `deliveredMass`.

Both records validate cleanly (the mechanized floor `validate_record()`
checks only non-emptiness). Read your own filled-in fields, not just this
outcome: is `AS-C06-EX`'s own `counterevidence` a specific competing design
and a specific unmodeled relation, the way real `AS-C06`'s names a
valve-controlled burner and Joule heating, or does it read like a hedge
that names nothing checkable? Does `AS-C08-EX`'s own `counterevidence` cite
specific, confirmed toolchain limits (your own D-029/D-030/D-031 findings)
the way real `AS-C08`'s does? Read each record's own `residual_uncertainties`
the same way: does it name a specific, checkable follow-up (what would let
the selection be revisited, or what the proof does not establish), or a
vague acknowledgment that "more work could be done"? And read each record's
own `premises` (printed above) against what its `rationale` actually rests
on: does `AS-C06-EX`'s own rationale actually use each premise, not just
list it?

In [ ]:
def word_count(text: str) -> int:
    return len(text.split())


def word_label(text: str) -> str:
    n = word_count(text)
    return f"{n} word" if n == 1 else f"{n} words"

for record in (as_c06_ex, as_c08_ex):
    print(f"{record.identifier}:")
    print(f"  counterevidence ({word_label(record.counterevidence)}): {record.counterevidence[:80]}...")
    print(f"  residual_uncertainties ({word_label(record.residual_uncertainties)}): {record.residual_uncertainties[:80]}...")
    print(f"  disposition: {record.disposition}")
    print(f"  engineering_conclusion: {record.engineering_conclusion}")


In [ ]:
# Your sufficiency reading here: for AS-C06-EX and AS-C08-EX, write out your
# own judgment -- is each record's claim, premises, counterevidence and
# residual_uncertainties substantive enough to trust (not a hedge that names
# nothing checkable), and does each record's own engineering_conclusion
# honestly match what its own evidence actually supports?
my_sufficiency_reading = """
# Your solution here
"""
print(my_sufficiency_reading)

A structural check alone cannot tell substantive text from a placeholder
that happens to be non-empty. The record below has every field
`validate_record()` requires, populated with text that would pass it
outright.

In [ ]:
placeholder_record = ReviewRecord(
    identifier="AS-PLACEHOLDER-EX",
    kind="asserted_solution",
    claim="The chosen design meets its requirement.",
    model_ref="CoffeeDemo",
    content_hash=hash_content(source),
    scope="CoffeeDemo",
    criteria="The design meets its requirement.",
    rationale="Analysis shows it does.",
    counterevidence="None known.",
    residual_uncertainties="None.",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

errors = validate_record(placeholder_record)
print(f"validate_record: errors={errors}")
print(f"counterevidence: {placeholder_record.counterevidence!r} ({word_label(placeholder_record.counterevidence)})")
print(f"residual_uncertainties: {placeholder_record.residual_uncertainties!r} ({word_label(placeholder_record.residual_uncertainties)})")
assert errors == [], "validate_record accepts this outright: every required field is non-empty"


`validate_record()` accepts this outright, the same as `AS-C06-EX` and
`AS-C08-EX` above: the mechanized floor only checks non-emptiness, and
"None known." and "None." both satisfy it. A sufficiency reading must
reject it anyway: neither field names a specific design alternative, a
specific unmodeled relation, or a specific toolchain limit the way a real,
substantive counterevidence field does; there is nothing here a reader
could go check, disagree with, or find wrong.

`AS-C06-EX`'s `engineering_conclusion` is fixed at `"undetermined"`;
`AS-C08-EX`'s is fixed at `"supported"`: the same two values real Chapter
9 finds honestly earned for `AS-C06` and `AS-C08`: an inductive,
domain-premise-argued mechanism selection stays `undetermined`, honestly,
while a claim already narrowed to exactly what a Z3 proof established is
`supported`. Whether YOUR OWN reconstruction earns those same two
conclusions depends on what your own `claim`, `counterevidence` and
`residual_uncertainties` actually say. Read them the way this step reads
the placeholder record above, not just by checking that they are
non-empty.

## Step 3: Stale detection, at scale

This step checks both records above together against the real, current
model in one pass, before and after one real edit. It reuses `as_c06_ex`
and `as_c08_ex` exactly as Step 2 built them, rather than rebuilding them a
second time (the same "reuse, don't rebuild" convention
`exercises/ch08/exercise.ipynb`'s own Step 3 already uses for `AS-C08-EX`).

A record with an empty identifier fails validation outright, distinct
from staleness: it was never a valid record to begin with, regardless of
which model it references.

In [ ]:
broken = ReviewRecord(
    identifier="",        # intentionally empty
    kind="asserted_solution",
    claim="Some claim",
    model_ref="CoffeeDemo",
    content_hash=hash_content(source),
    scope="Some scope",
    criteria="Some criteria",
    rationale="Some rationale",
    counterevidence="Some counterevidence",
    record_kind="worked_example",
)
errors = validate_record(broken)
assert len(errors) > 0, "Expected validation errors for empty identifier"
print(f"Negative control ok: errors={errors}")


Checked against your own model before any edit: `AS-C06-EX` was hashed
against `ch06_source` (your own earlier ch06 Step 2 model state), and
`AS-C08-EX` against `source` (your current Chapters 1-8 model) -- the same
two-file split real Chapter 9 uses. `AS-C06-EX` is already stale, from real
growth between when it was written (Chapter 6: `WaterMover` with an unbound
`throughput`) and the model this exercise's own Step 1 loads (Chapter 7
added `transferEfficiency` and `deliveredMass` to `WaterMover`, squarely
inside the scope `AS-C06-EX` itself declares,
`CoffeeDemo::WaterMover and its realizations`). `AS-C08-EX` was written
against this exact file (Chapter 8), so it is still current.

In [ ]:
from toaster.evidence import check_stale

tracked = {"AS-C06-EX": as_c06_ex, "AS-C08-EX": as_c08_ex}

print("Before any edit:")
for identifier, record in tracked.items():
    print(f"  {identifier}: stale={check_stale(record, source)}")

assert check_stale(as_c06_ex, source) is True
assert check_stale(as_c08_ex, source) is False

`AS-C06-EX`'s own `counterevidence` (yours, if it mirrors real `AS-C06`'s
own shape) names something specific as still missing about `WaterMover`'s
mechanism. A direct diff between `ch06_source` and the current model shows
real growth in the meantime, not just an unrelated bookkeeping change:
Chapter 7 added `transferEfficiency`, `transferEfficiencyBounded` and
`deliveredMass` to `WaterMover`, squarely inside the scope `AS-C06-EX`
itself declares (`CoffeeDemo::WaterMover and its realizations`).

In [ ]:
import difflib

ch06_lines = ch06_source.splitlines()
ch08_lines = source.splitlines()
added = [
    line for line in difflib.unified_diff(ch06_lines, ch08_lines, lineterm="")
    if line.startswith("+") and not line.startswith("+++")
]
transfer_efficiency_lines = [line for line in added if "transferefficiency" in line.lower()]
print(f"{len(transfer_efficiency_lines)} added lines mention transferEfficiency, including:")
for line in transfer_efficiency_lines[:3]:
    print(f"  {line.strip()}")

assert transfer_efficiency_lines, "Expected transferEfficiency-related growth between ch06 and the current model"

Loosen `deliveredMassBoundedBySupply`'s own bound (the companion lemma
copy `AS-C08-EX` is actually about), not `WaterMover`'s own
`transferEfficiencyBounded` or `deliveredMass` (the elements a real
`residual_uncertainties`, mirroring `AS-C08`'s own, would name as the risk
to watch): a different part of the same file, the same edit shape real
Chapter 9's own notebook 03 uses on the toaster's `deliveredEnergyBoundedBySupply`.

In [ ]:
revised_source = source.replace(
    "moverCheck.transferEfficiency <= 1.0",
    "moverCheck.transferEfficiency <= 1.2",
)
assert revised_source != source, "Expected the replacement to change the source"
assert "0.0 <= transferEfficiency and transferEfficiency <= 1.0" in revised_source, (
    "transferEfficiencyBounded's own bare-`transferEfficiency` bound must be untouched by this edit"
)
revised_model = conn.load_from_content(revised_source, strict=False)
assert revised_model.ok, "Revised model should still parse"

print("After loosening deliveredMassBoundedBySupply's own bound (1.0 -> 1.2):")
for identifier, record in tracked.items():
    print(f"  {identifier}: stale={check_stale(record, revised_source)}")

assert check_stale(as_c06_ex, revised_source) is True
assert check_stale(as_c08_ex, revised_source) is True
conn.close()

Both records now report stale, and the two histories genuinely match real
Chapter 9's own toaster pair: `AS-C06-EX` was already stale before this
edit, for a reason entirely unrelated to it (Chapter 7's real growth,
confirmed by the diff above); `AS-C08-EX` was current before the edit and
only went stale because of it.

`AS-C06-EX`'s own `scope` (`CoffeeDemo::WaterMover and its realizations`)
is a judgment about which mechanism `WaterMover` commits to.
`deliveredMassBoundedBySupply` is a package-level lemma `WaterMover` does
not own; it references a `WaterMover`-typed usage (`moverCheck`) only
incidentally, as the Chapter 8 companion-lemma pattern requires.
`AS-C06-EX`'s own staleness has nothing to do with this specific edit: it
was already stale beforehand, from real growth the diff above confirms,
squarely inside its own declared scope.

`AS-C08-EX`'s own `model_ref` names `deliveredMassBoundedBySupply`
directly: this edit hits exactly the construct the record is about. But
if your own `residual_uncertainties` mirrors real `AS-C08`'s, it names a
DIFFERENT specific risk: editing `WaterMover`'s own
`transferEfficiencyBounded` or `deliveredMass` (the ORIGINAL construct),
not the restated companion-lemma copy this edit actually touches. Read
that way, `AS-C08-EX` goes stale for a reason adjacent to, but not
identical to, what its own residual specifically names, caught only
because `content_hash` covers the whole file, exactly mirroring real
`AS-C08`'s own story in `03-stale-detection.ipynb`.

Checking either record alone, at either point, would only have reported
"stale" or "current," not the reason -- the same point real Chapter 9's
own notebook 03 makes, this time reproduced for real: two genuinely
different histories, one substantively overtaken by real growth (caught
before the edit), the other caught only by a coarse, whole-file safeguard
after a targeted edit.